# Trabalho 2 — Otimização Multiobjetivo (experimento reformulado)
## Estudo do *knee-point*: NSGA-II × R-NSGA-II × PI-NSGA-II

**Integrantes:** _(preencher)_

---

### Desenho do experimento
Para cada problema (**Truss2D**, **WeldedBeam** — 2 objetivos; **Carside** — 3 objetivos):

1. Roda-se o **NSGA-II 21 vezes** (sementes independentes).
2. Cada frente é **normalizada para [0,1]** em todos os eixos (mesmo referencial do experimento anterior).
3. Calcula-se o **Hipervolume** de cada execução (HV sobre a frente normalizada) — **só do NSGA-II**.
4. A execução de **HV mediano** (11ª de 21) é o *padrão ouro*.
5. Nessa frente, o **knee** é o ponto de **máxima distância à reta/plano que liga os extremos** (2D → reta, 3D → plano).

Esse **knee mediano do NSGA-II** é o objeto de comparação, injetado de duas formas:
- **R-NSGA-II** → entra como `ref_points` (atração por distância normalizada);
- **PI-NSGA-II** → entra como **alvo fixo** de um decisor automático (`KneePointDM`), que no
  modo *pairwise* nativo do pymoo prefere a solução mais próxima do knee.

O **ponto ideal** permanece nos gráficos como referência visual, mas **fora** da métrica.

### Robustez do PI-NSGA-II
O PI-NSGA-II ajusta uma **função de valor** a cada `tau` gerações; em alguns casos esse ajuste
(`create_poly_vf`) falha e quebra a execução. Usamos o **`SafePINSGA2`**: protege a lógica de
preferência num `try/except` — se o ajuste falha, aquela geração vira NSGA-II puro por um ciclo
(contado em `n_fallbacks`) em vez de matar a execução.

### Saídas
Dois CSVs **irmãos** com os pontos das frentes (uma linha por solução):
- `solucoes_brutas.csv` — valores das funções objetivo (escala original);
- `solucoes_normalizadas.csv` — os mesmos pontos no referencial **[0,1]** do NSGA-II.

> Custo: o PI-NSGA-II domina o tempo. Comece com `MODO_TESTE = True` para validar o pipeline.


## 1. Imports e configuração

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # registra projeção 3D

from pymoo.problems.multi import Carside, Truss2D, WeldedBeam
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.algorithms.moo.rnsga2 import RNSGA2
from pymoo.algorithms.moo.pinsga2 import PINSGA2, AutomatedDM
from pymoo.util.reference_direction import select_points_with_maximum_distance
from pymoo.optimize import minimize
from pymoo.indicators.hv import HV

# ----------------------------------------------------------------------
# CONFIGURAÇÃO (knobs principais)
# ----------------------------------------------------------------------
MODO_TESTE = False                  # True -> roda rápido só p/ checar o pipeline
N_RUNS  = 3   if MODO_TESTE else 21
N_GEN   = 40  if MODO_TESTE else 200
POP_SIZE = 100
HV_REF  = 1.1                      # ponto de referência do HV no espaço normalizado [0,1]^k

PROBLEMAS = {
    "Truss2D":    Truss2D(),       # 2 objetivos
    "WeldedBeam": WeldedBeam(),    # 2 objetivos
    "Carside":    Carside(),       # 3 objetivos
}

## 2. Funções de apoio: normalização, knee e HV

O **knee** é definido de forma geral pela distância ao hiperplano que passa pelos pontos extremos
(melhores em cada objetivo): em 2D esse hiperplano é uma reta; em 3D, um plano. O knee é o ponto
que mais "afunda" em direção ao ideal.

In [2]:
def normalizar(F, ideal, nadir):
    """Normaliza F para [0,1] em cada objetivo usando ideal/nadir de referência."""
    F = np.atleast_2d(np.asarray(F, float))
    den = nadir - ideal
    den = np.where(den == 0, 1.0, den)
    return (F - ideal) / den

def encontrar_knee(Fn):
    """Índice do knee: máxima distância ao hiperplano dos extremos (2D reta / 3D plano)."""
    k = Fn.shape[1]
    ext = np.array([Fn[np.argmin(Fn[:, j])] for j in range(k)])   # k pontos extremos
    try:
        n = np.linalg.solve(ext, np.ones(k))   # hiperplano  n . x = 1
        idx = int(np.argmin(Fn @ n))           # knee = ponto mais "abaixo" do plano
    except np.linalg.LinAlgError:
        idx = 0
    return idx

def hv_normalizado(Fn, k):
    """Hipervolume da frente normalizada, ref point = (HV_REF,...,HV_REF)."""
    return float(HV(ref_point=np.full(k, HV_REF))(Fn))

def registrar_pontos(nome, alg, run, F, destino):
    """Acumula os pontos de uma frente no formato do notebook de análise."""
    for p in F:
        destino.append({"problema": nome, "algoritmo": alg, "run": run,
                        "f1": p[0], "f2": p[1], "f3": (p[2] if len(p) == 3 else np.nan)})

## 3. Decisor automático + PI-NSGA-II blindado

`KneePointDM` persegue um **alvo fixo** = o knee mediano do NSGA-II (em espaço normalizado, para
ficar no mesmo referencial e não sofrer com as escalas diferentes dos objetivos do Carside).
`SafePINSGA2` mantém a lógica original do PI-NSGA-II, só que protegida: **não** sobrescreve o alvo
com o knee da frente corrente e captura falhas do ajuste da função de valor.

In [3]:
from pymoo.algorithms.moo.pinsga2 import PINSGA2, AutomatedDM
from pymoo.util.reference_direction import select_points_with_maximum_distance

class KneePointDM(AutomatedDM):
    """Prefere a solução mais próxima de um ALVO FIXO = knee mediano do NSGA-II.
       Distância medida no espaço normalizado [0,1] (referencial do NSGA-II)."""
    def __init__(self, knee_ref_norm, ideal, nadir):
        super().__init__()
        self.target = np.asarray(knee_ref_norm, float)
        self.ideal, self.nadir = ideal, nadir

    def _norm(self, F):
        den = self.nadir - self.ideal
        den = np.where(den == 0, 1.0, den)
        return (np.atleast_2d(F) - self.ideal) / den

    def makeDecision(self, F):
        d = np.linalg.norm(self._norm(F) - self.target, axis=1)   # F = 2 pontos
        if d[0] < d[1]:   return 'a'
        elif d[1] < d[0]: return 'b'
        else:             return 'c'


class SafePINSGA2(PINSGA2):
    """PI-NSGA-II blindado + higiene de memória:
       (1) NÃO acumula historical_F entre gerações;
       (2) limita o custo do ajuste da função de valor (max_iter finito);
       protege a lógica de preferência num try/except."""
    def _advance(self, infills=None, **kwargs):
        from pymoo.algorithms.base.genetic import GeneticAlgorithm
        import pymoo.util.value_functions as mvf

        GeneticAlgorithm._advance(self, infills=infills, **kwargs)

        try:
            rank, F = self.pop.get("rank", "F")
            self.fronts = rank
            frente = F[rank == 0]
            if frente.shape[0] == 0:
                return

            # (MUDANÇA 1) NÃO acumular histórico: guarda só a frente atual.
            # Antes: self.historical_F = np.vstack((self.historical_F, frente)) -> crescia sem limite.
            self.historical_F = frente

            to_find = self.eta if frente.shape[0] >= self.eta else frente.shape[0]
            if self.presi_signs is None:
                self.presi_signs = np.ones(frente.shape[1])
            if to_find == 0:
                return

            eta_idx = select_points_with_maximum_distance(frente, to_find, random_state=self.random_state)
            self.eta_F = frente[eta_idx]
            self.eta_F = self.eta_F[self.eta_F[:, 0].argsort()]
            self.eta_F = np.unique(self.eta_F, axis=0)
            self.paused_F = frente
            self.prev_pop = self.pop

            if self.n_gen % self.tau != 0:
                return
            if len(self.eta_F) < 2:
                self._reset_dm_preference()
                return

            dm_ranks = (self.automated_dm.makePairwiseDecision(self.eta_F)
                        if self.automated_dm else
                        PINSGA2._get_pairwise_ranks(self.eta_F, self.presi_signs))
            if len(set(dm_ranks)) == 0:
                self._reset_dm_preference()
                return

            eta_F = self.eta_F
            while eta_F.shape[0] > 1:
                # (MUDANÇA 2) limita o ajuste do scipy com max_iter finito (self.vf_max_iter).
                vf_res = mvf.create_poly_vf(eta_F * -1, dm_ranks.tolist(),
                                            eps_max=self.eps_max, method=self.opt_method,
                                            max_iter=self.vf_max_iter)
                if vf_res.fit:
                    self.vf_res = vf_res
                    self.vf_plot_flag = True
                    self.v2 = self.vf_res.vf(eta_F[dm_ranks[1] - 1] * -1).item()
                    break
                else:
                    if eta_F.shape[0] == 2:
                        self._reset_dm_preference()   # (opção a) volta a NSGA-II puro neste ciclo
                        break
                    rt = dm_ranks[1]
                    eta_F = np.delete(eta_F, rt - 1, axis=0)
                    dm_ranks = np.concatenate(([dm_ranks[0]], dm_ranks[2:]))
                    dm_ranks[dm_ranks > rt] -= 1

        except Exception:
            self.n_fallbacks = getattr(self, "n_fallbacks", 0) + 1
            try:
                self._reset_dm_preference()
            except Exception:
                pass
            return

print("KneePointDM e SafePINSGA2 (com higiene de memória) definidos")

KneePointDM e SafePINSGA2 (com higiene de memória) definidos


## 4. Padrão ouro: NSGA-II (21 execuções → knee de referência)

Estabelece, por problema: o referencial de normalização (`ideal`/`nadir`), o **knee de referência**
(execução de HV mediano) e acumula os pontos brutos do NSGA-II.

In [4]:
PONTOS = []           # todos os pontos brutos (as 3 execuções acumulam aqui)
referencias = {}      # knee de referência + ideal/nadir por problema

for nome, problema in PROBLEMAS.items():
    k = problema.n_obj
    fronts = [minimize(problema, NSGA2(pop_size=POP_SIZE), ("n_gen", N_GEN),
                       seed=s, verbose=False).F for s in range(N_RUNS)]
    ideal, nadir = np.vstack(fronts).min(0), np.vstack(fronts).max(0)

    hvs = [hv_normalizado(normalizar(F, ideal, nadir), k) for F in fronts]   # HV só do NSGA-II
    med = int(np.argsort(hvs)[N_RUNS // 2])                                   # execução mediana
    Fmed_n = normalizar(fronts[med], ideal, nadir)
    ki = encontrar_knee(Fmed_n)
    referencias[nome] = dict(ideal=ideal, nadir=nadir,
                             knee_norm=Fmed_n[ki], knee_raw=fronts[med][ki])

    for s, F in enumerate(fronts):
        registrar_pontos(nome, "NSGA-II", s, F, PONTOS)
    print(f"{nome}: NSGA-II ok | knee ref (norm) = {np.round(Fmed_n[ki], 3)}")

Truss2D: NSGA-II ok | knee ref (norm) = [0.192 0.182]
WeldedBeam: NSGA-II ok | knee ref (norm) = [0.196 0.083]
Carside: NSGA-II ok | knee ref (norm) = [0.322 0.609 0.405]


## 5. R-NSGA-II (guiado pelo knee, em objetivos brutos)

O knee de referência entra como `ref_points`; o R-NSGA-II normaliza internamente
(via `calc_norm_pref_distance`), então passamos o knee em escala **original**.

In [5]:
for nome, problema in PROBLEMAS.items():
    ref = referencias[nome]
    for s in range(N_RUNS):
        algo = RNSGA2(ref_points=np.array([ref["knee_raw"]]), pop_size=POP_SIZE, epsilon=0.01)
        F = minimize(problema, algo, ("n_gen", N_GEN), seed=s, verbose=False).F
        registrar_pontos(nome, "R-NSGA-II", s, F, PONTOS)
    print(f"{nome}: R-NSGA-II ok")

Truss2D: R-NSGA-II ok
WeldedBeam: R-NSGA-II ok
Carside: R-NSGA-II ok


## 6. PI-NSGA-II blindado (guiado pelo mesmo knee)

O `KneePointDM` recebe o knee **normalizado** de referência; o `n_fallbacks` reportado é o
termômetro de robustez (quantas gerações caíram no modo NSGA-II por falha do ajuste).

In [6]:
import gc

for nome, problema in PROBLEMAS.items():
    ref = referencias[nome]
    nfb = 0
    for s in range(N_RUNS):
        dm = KneePointDM(ref["knee_norm"], ref["ideal"], ref["nadir"])
        algo = SafePINSGA2(pop_size=POP_SIZE, automated_dm=dm,
                           tau=30, eta=4, vf_max_iter=300, verbose=False)
        F = minimize(problema, algo, ("n_gen", N_GEN), seed=s, verbose=False).F
        nfb += getattr(algo, "n_fallbacks", 0)
        registrar_pontos(nome, "PI-NSGA-II", s, F, PONTOS)

        # (MUDANÇA 3) libera memória entre execuções (evita acúmulo de objetos mortos)
        del algo, dm, F
        gc.collect()
    print(f"{nome}: PI-NSGA-II ok | fallbacks totais = {nfb}")

Truss2D: PI-NSGA-II ok | fallbacks totais = 0
WeldedBeam: PI-NSGA-II ok | fallbacks totais = 0
Carside: PI-NSGA-II ok | fallbacks totais = 0


## 7. Salvamento: dois CSVs irmãos (bruto + normalizado)

`solucoes_brutas.csv` já está no formato que o notebook de análise lê
(`problema, algoritmo, run, f1, f2, f3`). O normalizado usa o **mesmo referencial** do NSGA-II
por problema — alguns pontos do R/PI podem cair um pouco fora de [0,1], o que é esperado.

In [7]:
# CSV bruto (valores das funções objetivo, escala original)
df_bruto = pd.DataFrame(PONTOS)
df_bruto.to_csv("solucoes_brutas.csv", index=False)
print("salvo: solucoes_brutas.csv", df_bruto.shape)

# CSV normalizado (mesmos pontos, referencial [0,1] do NSGA-II por problema)
partes = []
for nome in PROBLEMAS:
    ref = referencias[nome]; k = int(PROBLEMAS[nome].n_obj)
    sub = df_bruto[df_bruto["problema"] == nome].copy()
    cols = ["f1", "f2"] + (["f3"] if k == 3 else [])
    sub[cols] = normalizar(sub[cols].values, ref["ideal"], ref["nadir"])
    partes.append(sub)
df_norm = pd.concat(partes, ignore_index=True)
df_norm.to_csv("solucoes_normalizadas.csv", index=False)
print("salvo: solucoes_normalizadas.csv", df_norm.shape)

display(df_bruto.head())
display(df_norm.head())

salvo: solucoes_brutas.csv (18900, 6)
salvo: solucoes_normalizadas.csv (18900, 6)


,problema,algoritmo,run,f1,f2,f3
0,Truss2D,NSGA-II,0,0.053167,8432.740432,NaN
1,Truss2D,NSGA-II,0,0.004040,99997.034072,NaN
2,Truss2D,NSGA-II,0,0.023997,16712.174490,NaN
3,Truss2D,NSGA-II,0,0.050184,8534.919995,NaN
4,Truss2D,NSGA-II,0,0.046566,8809.550667,NaN


,problema,algoritmo,run,f1,f2,f3
0,Truss2D,NSGA-II,0,0.788552,5.298761e-11,NaN
1,Truss2D,NSGA-II,0,0.000496,9.999834e-01,NaN
2,Truss2D,NSGA-II,0,0.320626,9.042058e-02,NaN
3,Truss2D,NSGA-II,0,0.740692,1.115914e-03,NaN
4,Truss2D,NSGA-II,0,0.682653,4.115185e-03,NaN


## 8. Notas para a documentação

- **Normalização única por problema:** todos os algoritmos usam o `ideal`/`nadir` do NSGA-II,
  então tudo vive no mesmo referencial $[0,1]^k$ e é comparável.
- **knee em 3D (Carside):** mesma definição geral (distância ao plano dos extremos), sem caso
  especial de código além de `n_obj = 3`.
- **`SafePINSGA2` / `n_fallbacks`:** se os fallbacks vierem altos, o PI está "virando NSGA-II"
  com frequência — vale revisar `tau`/`eta` ou o decisor.
- **Alvo do PI = knee do NSGA-II:** o mesmo alvo que guia o R-NSGA-II, deixando os três
  algoritmos comparáveis contra a mesma referência.
- **Análise/visualização:** aponte o notebook de análise para `solucoes_brutas.csv` — as colunas
  já batem (`problema, algoritmo, run, f1, f2, f3`).
